# Test graph

In [ ]:

import gc
import os
import re
import json
from pathlib import Path
from typing import Any
import random
import pandas as pd

from dotenv import load_dotenv

# load env
load_dotenv()


In [ ]:
DATA_DIR = Path("../data")
print(f"DATA_DIR: {DATA_DIR.resolve()}")

GRAPH_DIR = DATA_DIR / "graph"


HAS_SAVE_JSON_GRAPH = False

In [ ]:
from src.graph.builder_graph import BuilderGraph
from src.graph.graph_repository import LegalGraphRepository
from src.graph.graph_service import LegalGraphService
from src.graph.neo4j_client import Neo4jClient
from src.utils.logger import setup_logger

print("Starting test_venv.py..." + os.environ.get("MODE", ""))

class TestGraph:
    def __init__(self, service: LegalGraphService):
        self.logger = setup_logger(name="test_venv", filename="test_venv.log")
        self.logger.info("TestVenv initialized.")
        self.builder_graph = BuilderGraph()
        self.legal_service = service

    def test_import(self):
        self.logger.info("Ok import of modules...")
        self.builder_graph.add_node("node1", {"data": "value1"})
        
    def test_graph(self):
        self.logger.info("Testing graph service...")
        
        # nodes
        legal_nodes = self.legal_service.get_all_legal_nodes(limit=5000)
        self.logger.info(f"Retrieved {len(legal_nodes)} legal nodes.")
        
        # 10 first nodes
        for i, node in enumerate(legal_nodes[:10]):
            self.logger.info(f"Node {i+1}: {node}")
        
        # relationships
        legal_relationships = self.legal_service.get_all_legal_relationships(limit=5000)
        self.logger.info(f"Retrieved {len(legal_relationships)} legal relationships.")
        
        # 10 first relationships
        for i, relationship in enumerate(legal_relationships[:10]):
            self.logger.info(f"Relationship {i+1}: {relationship}")
        
        gc.collect()  # Force garbage collection to free up memory
        
print(os.environ.get("NEO4J_URI"))
neo4j_client = Neo4jClient(
    uri=os.environ.get("NEO4J_URI"),
    username=os.environ.get("NEO4J_USERNAME"),
    password=os.environ.get("NEO4J_PASSWORD"),
    database=os.environ.get("NEO4J_DATABASE")
)
repo = LegalGraphRepository(db=neo4j_client)
service = LegalGraphService(repository=repo)
test_graph = TestGraph(service=service)
test_graph.test_import()
test_graph.test_graph()

In [ ]:
def save_to_json(data: Any, filename: str | Path):
    """Save data to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    print(f"Data saved to {filename}")

In [ ]:
client  = Neo4jClient()                       # đọc NEO4J_URI/USERNAME/PASSWORD/DATABASE từ .env
repo    = LegalGraphRepository(db=client)
service = LegalGraphService(repository=repo)

if HAS_SAVE_JSON_GRAPH:

    save_to_json(service.get_all_documents(), GRAPH_DIR / "all_documents.json")
    save_to_json(service.get_all_legal_nodes(limit=10000), GRAPH_DIR / "all_legal_nodes.json")
    save_to_json(service.get_node_by_id("BLDS_2015_D27_K1_DIEM_A"), GRAPH_DIR / "BLDS_2015_D27_K1_DIEM_A.json")

    save_to_json(
        service.resolve_target_node("BLDS_2015", {"chapter": "III", "article": "27", "clause": "1", "point": "a"}), 
        GRAPH_DIR / "BLDS_2015_resolve_target_node.json"
    )
    save_to_json(
        service.get_breadcrumbs("BLDS_2015_D27_K1"), 
        GRAPH_DIR / "BLDS_2015_breadcrumbs.json"
    )

    # service.get_children("BLDS_2015_D27")         # các Khoản của Điều 27
    save_to_json(service.get_children("BLDS_2015_D27"), GRAPH_DIR / "BLDS_2015_D27_children.json")
    # service.get_descendants("BLDS_2015_P1_CH3", max_depth=5, limit=200)
    save_to_json(service.get_descendants("BLDS_2015_P1_CH3", max_depth=5, limit=200), GRAPH_DIR / "BLDS_2015_P1_CH3_descendants.json")
    # service.get_related_laws("BLDS_2015_D27_K1")  # REFERENCES/IMPLEMENTS 1-hop
    save_to_json(service.get_related_laws("BLDS_2015_D27_K1"), GRAPH_DIR / "BLDS_2015_D27_K1_related_laws.json")
    # service.get_document_provenance("BLDS_2015")  # số hiệu, hiệu lực, file nguồn
    save_to_json(service.get_document_provenance("BLDS_2015"), GRAPH_DIR / "BLDS_2015_provenance.json")
    # service.get_graph_summary()                   # thống kê node/relation theo loại
    save_to_json(service.get_graph_summary(), GRAPH_DIR / "graph_summary.json")

# Truy vấn luật với Graph từ input

In [ ]:

import gc
import os
import re
import json
from pathlib import Path
from typing import Any
import random
import pandas as pd

from dotenv import load_dotenv

# load env
load_dotenv()


## Cấu hình và đương dẫn file

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


ROOT_DIR = Path("../").resolve()
print(f"ROOT_DIR: {ROOT_DIR}")

DATA_DIR = ROOT_DIR / "data"

RETRIEVAL_DIR = DATA_DIR / "retrieval"


## Input truy vấn mẫu
- Các document liên quan được cung cấp sẵn (Semantic search để tìm kiếm document thu hẹp phạm vi trước)

In [44]:
QUERY_INPUTS = [ 
    {
        "task_id": "LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION",
        "input": "Nội dung Điều 1 theo Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 là gì?",
        "output": {
            "part": None,
            "chapter": None,
            "section": None,
            "subsection": None,
            "article": "1",
            "clause": None,
            "point": None,
            "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
        }
    },
    {
        "task_id": "LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION",
        "input": "Điều 1 thuộc Chương I  theo Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 quy định gì?",
        "output": {
            "part": None,
            "chapter": "I",
            "section": None,
            "subsection": None,
            "article": "1",
            "clause": None,
            "point": None,
            "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
        }
    },
    {
        "task_id": "LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION",
        "input": "Điều 2 của Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 quy định về vấn đề gì?",
        "output": {
            "part": None,
            "chapter": None,
            "section": None,
            "subsection": None,
            "article": "2",
            "clause": None,
            "point": None,
            "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
        }
    },
]

RELATED_DOCUMENTS = [
    {
        "verification_scope": "effective_from only; status and effective_to are unverified legacy values",
        "label": "Document",
        "document_id": "BLDS_2015",
        "title": "Bộ luật Dân sự 2015",
        "issuer": "Quốc hội",
        "source_file": "bo_luat_dan_su_91-2015-QH13.docx",
        "number": "91/2015/QH13",
        "metadata_source": "DOCX nguồn: van_ban/bo_luat_dan_su_91-2015-QH13.docx; effective_from: Điều 689",
        "verified_at": "2026-10-07",
        "effective_from": "2017-01-01",
        "node_id": "BLDS_2015",
        "document_type": "BO_LUAT",
        "status": "CON_HIEU_LUC",
        "labels": [
            "LegalNode",
            "Document"
        ]
    },
    {
        "verification_scope": "effective_from verified via NQ 64/2013/QH13 (ItemID 32804); status verified via CSDL Quốc gia VBPL ItemID 32801 (sửa đổi bởi NQ 203/2025/QH15)",
        "status_verified_at": "2026-10-07",
        "label": "Document",
        "document_id": "HIEN_PHAP_2013",
        "title": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013",
        "issuer": "Quốc hội",
        "source_file": "Hien_phap_nam_2013.docx",
        "source_url": "https://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=32801",
        "number": "Không số",
        "metadata_source": "DOCX nguồn: van_ban/Hien_phap_nam_2013.docx; effective_from: Nghị quyết 64/2013/QH13, Điều 1; status: CSDL Quốc gia VBPL ItemID 32801 (sửa đổi bởi Nghị quyết 203/2025/QH15)",
        "effective_from_source_url": "https://vbpl.vn/TW/Pages/vbpq-toanvan.aspx?ItemID=32804",
        "verified_at": "2026-10-07",
        "effective_from": "2014-01-01",
        "node_id": "HIEN_PHAP_2013",
        "document_type": "HIEN_PHAP",
        "status": "HET_HIEU_LUC_MOT_PHAN",
        "labels": [
            "LegalNode",
            "Document"
        ]
    },
    # {
    #     "verification_scope": "effective_from only; status and effective_to are unverified legacy values",
    #     "label": "Document",
    #     "document_id": "ND_19_2019",
    #     "title": "Nghị định 19/2019/NĐ-CP về họ, hụi, biêu, phường",
    #     "issuer": "Chính phủ",
    #     "source_file": "19_2019_ND-CP_386324.docx",
    #     "number": "19/2019/NĐ-CP",
    #     "metadata_source": "DOCX nguồn: van_ban/19_2019_ND-CP_386324.docx; effective_from: Điều 26 khoản 1",
    #     "verified_at": "2026-10-07",
    #     "effective_from": "2019-04-05",
    #     "node_id": "ND_19_2019",
    #     "document_type": "NGHI_DINH",
    #     "status": "CON_HIEU_LUC",
    #     "labels": [
    #         "LegalNode",
    #         "Document"
    #     ]
    # },
    # {
    #     "verification_scope": "effective_from only; status and effective_to are unverified legacy values",
    #     "label": "Document",
    #     "document_id": "ND_21_2021",
    #     "title": "Nghị định 21/2021/NĐ-CP quy định thi hành Bộ luật Dân sự về bảo đảm thực hiện nghĩa vụ",
    #     "issuer": "Chính phủ",
    #     "source_file": "21_2021_ND-CP_468069.docx",
    #     "number": "21/2021/NĐ-CP",
    #     "metadata_source": "DOCX nguồn: van_ban/21_2021_ND-CP_468069.docx; effective_from: Điều 60 khoản 1",
    #     "verified_at": "2026-10-07",
    #     "effective_from": "2021-05-15",
    #     "node_id": "ND_21_2021",
    #     "document_type": "NGHI_DINH",
    #     "status": "CON_HIEU_LUC",
    #     "labels": [
    #         "LegalNode",
    #         "Document"
    #     ]
    # },
    # {
    #     "verification_scope": "effective_from only; status and effective_to are unverified legacy values",
    #     "label": "Document",
    #     "document_id": "NQ_02_2022",
    #     "title": "Nghị quyết 02/2022/NQ-HĐTP hướng dẫn áp dụng một số quy định của Bộ luật Dân sự về trách nhiệm bồi thường thiệt hại ngoài hợp đồng",
    #     "issuer": "Hội đồng Thẩm phán Tòa án nhân dân tối cao",
    #     "source_file": "02_2022_NQ-HDTP_483799.docx",
    #     "number": "02/2022/NQ-HĐTP",
    #     "metadata_source": "DOCX nguồn: van_ban/02_2022_NQ-HDTP_483799.docx; effective_from: Điều 13 khoản 1",
    #     "verified_at": "2026-10-07",
    #     "effective_from": "2023-01-01",
    #     "node_id": "NQ_02_2022",
    #     "document_type": "NGHI_QUYET",
    #     "status": "CON_HIEU_LUC",
    #     "labels": [
    #         "LegalNode",
    #         "Document"
    #     ]
    # }
]

## Cell chạy chạy Model hoặc Regex để lấy cấu trích trích dẫn

In [45]:
print("Chạy bóc tách cấu trúc")

Chạy bóc tách cấu trúc


## Kiểm trả tính hợp lệ của cấu trúc phân cấp đã trích xuất

In [46]:
from src.common.schemas import LegalHierarchy



for input_data in QUERY_INPUTS:
    print("--------------------------------------------------")
    query_text = input_data["input"]
    extracted_hierarchy = input_data["output"]
    
    print(f"Query: {query_text}")
    print(f"Extracted hierarchy: {json.dumps(extracted_hierarchy, ensure_ascii=False, indent=4)}")
    
    # LegalHierarchy
    legal_hierarchy = LegalHierarchy.from_dict(extracted_hierarchy)
    
    validation_result, validation_error = legal_hierarchy.validate(require_document=True)
    
    if not validation_result:
        print(f"Validation failed for query: {query_text}") 
        print(f"Error: {validation_error}")
        
        print("Chuyển sang cách bóc tách khác hoặc semantic.")
        continue
    
    print(f"Thông tin của tạo node: {json.dumps(legal_hierarchy.get_all_info(), ensure_ascii=False, indent=4)}")
        
    
    


--------------------------------------------------
Query: Nội dung Điều 1 theo Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 là gì?
Extracted hierarchy: {
    "part": null,
    "chapter": null,
    "section": null,
    "subsection": null,
    "article": "1",
    "clause": null,
    "point": null,
    "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013"
}
Thông tin của tạo node: {
    "dict": {
        "document": "Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013",
        "part": null,
        "chapter": null,
        "section": null,
        "subsection": null,
        "article": "1",
        "clause": null,
        "point": null
    },
    "node_type": "Article",
    "parent_node_type": "Document",
    "citation_path": "Điều 1 của Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013",
    "postfix_node_id": "_D1"
}
--------------------------------------------------
Query: Điều 1 thuộc Chương I  theo Hiến pháp nước Cộng hòa xã hội chủ n

In [47]:

for input_data in QUERY_INPUTS:

    for doc in RELATED_DOCUMENTS:
        print("--------------------------------------------------")
        print(f"Document: {doc['title']} ({doc['document_id']})") 

        doc_id = doc.get("document_id")


        query_text = input_data["input"]
        extracted_hierarchy = input_data["output"]

        print(f"Query: {query_text}")

        # LegalHierarchy
        legal_hierarchy = LegalHierarchy.from_dict(extracted_hierarchy)

        validation_result, validation_error = legal_hierarchy.validate(require_document=True)

        if not validation_result:
            print(f"Validation failed for query: {query_text}") 
            print(f"Error: {validation_error}")
            
            print("Chuyển sang cách bóc tách khác hoặc semantic.")
            continue

        # print(f"Thông tin của tạo node: {json.dumps(legal_hierarchy.get_all_info(), ensure_ascii=False, indent=4)}")

        node_type = legal_hierarchy.get_node_type()
        parent_node_type = legal_hierarchy.get_parent_node_type()
        citation_path = legal_hierarchy.get_citation_path()
        node_id = legal_hierarchy.get_node_id_from_doc_id(doc_id=doc_id)

        print(f"Node id: {node_id}")


        # Duyệt đồ thị
        node_info = {
            'information': service.get_node_by_id(node_id),
            'related_laws': service.get_related_laws(node_id),
            'children': service.get_children(node_id),
            'descendants': service.get_descendants(node_id, max_depth=5, limit=200),
            'relationships': service.get_relationships_by_node(node_id, limit=200)
        }

        for key, value in node_info.items():
            print(f"{key}: {json.dumps(value, ensure_ascii=False, indent=4)}")

2026-10-14 01:48:07 - [INFO] - graph_repository.py - Tìm node theo node_id='BLDS_2015_D1'...
2026-10-14 01:48:07 - [INFO] - graph_repository.py - Tìm node theo node_id='HIEN_PHAP_2013_D1'...
2026-10-14 01:48:07 - [INFO] - graph_repository.py - Tìm node theo node_id='BLDS_2015_D1'...
2026-10-14 01:48:07 - [INFO] - graph_repository.py - Tìm node theo node_id='HIEN_PHAP_2013_D1'...
2026-10-14 01:48:07 - [INFO] - graph_repository.py - Tìm node theo node_id='BLDS_2015_D2'...


--------------------------------------------------
Document: Bộ luật Dân sự 2015 (BLDS_2015)
Query: Nội dung Điều 1 theo Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam năm 2013 là gì?
Node id: BLDS_2015_D1
information: {
    "number": 1,
    "parent_id": "BLDS_2015_P1_CH1",
    "full_text": "Phạm vi điều chỉnh\nBộ luật này quy định địa vị pháp lý, chuẩn mực pháp lý về cách ứng xử của cá nhân, pháp nhân; quyền, nghĩa vụ về nhân thân và tài sản của cá nhân, pháp nhân trong các quan hệ được hình thành trên cơ sở bình đẳng, tự do ý chí, độc lập về tài sản và tự chịu trách nhiệm (sau đây gọi chung là quan hệ dân sự).",
    "label": "Article",
    "text": "Bộ luật này quy định địa vị pháp lý, chuẩn mực pháp lý về cách ứng xử của cá nhân, pháp nhân; quyền, nghĩa vụ về nhân thân và tài sản của cá nhân, pháp nhân trong các quan hệ được hình thành trên cơ sở bình đẳng, tự do ý chí, độc lập về tài sản và tự chịu trách nhiệm (sau đây gọi chung là quan hệ dân sự).",
    "document_id": "BLDS_2015

2026-10-14 01:48:07 - [INFO] - graph_repository.py - Tìm node theo node_id='HIEN_PHAP_2013_D2'...


information: {
    "number": 2,
    "parent_id": "HIEN_PHAP_2013_CH1",
    "full_text": "Khoản 1. Nhà nước Cộng hòa xã hội chủ nghĩa Việt Nam là nhà nước pháp quyền xã hội chủ nghĩa của Nhân dân, do Nhân dân, vì Nhân dân.\nKhoản 2. Nước Cộng hòa xã hội chủ nghĩa Việt Nam do Nhân dân làm chủ; tất cả quyền lực nhà nước thuộc về Nhân dân mà nền tảng là liên minh giữa giai cấp công nhân với giai cấp nông dân và đội ngũ trí thức.\nKhoản 3. Quyền lực nhà nước là thống nhất, có sự phân công, phối hợp, kiểm soát giữa các cơ quan nhà nước trong việc thực hiện các quyền lập pháp, hành pháp, tư pháp.",
    "label": "Article",
    "document_id": "HIEN_PHAP_2013",
    "node_id": "HIEN_PHAP_2013_D2",
    "labels": [
        "LegalNode",
        "Article"
    ]
}
related_laws: []
children: [
    {
        "number": 1,
        "parent_id": "HIEN_PHAP_2013_D2",
        "label": "Clause",
        "text": "Nhà nước Cộng hòa xã hội chủ nghĩa Việt Nam là nhà nước pháp quyền xã hội chủ nghĩa của Nhân dân, d

## Tìm các Document liên quan trong Graph

## Tạo các node id từ cấu trúc phân cấp

## Tìm kiếm các node liên quan và các relationship